**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**04 | Bounded Temporal CNN and Matched Random Forest**

![HARTH bounded temporal classification workflow](../../images/04_consolidated_workflow.svg)

**Objective:** Assess a compact temporal 1D-CNN and compare its Validation predictions with an established Random Forest on identical HARTH Development windows and participant partitions.

**Dataset:** Official pinned NTNU HARTH v2.0 with six acceleration channels at nominal 50 Hz. Eligible signal windows contain 100 continuous samples of one activity label, using a 50-sample stride. Up to 500 eligible windows are selected per Development participant, evenly along the original source order without label-based resampling.

**Design:** Build the bounded raw-signal cohort once, apply a participant-disjoint 70% Train, 20% Validation and 10% unused Development split, train the five-epoch CNN with Train-only channel scaling and compare against a 120-tree Random Forest using 30 statistical features computed from the same windows.

**Scientific boundary:** S023 and S032 remain outside all Development fitting and scoring. This bounded single-split comparison differs from full Development five-fold classical cross-validation. Matching source identities does not remove differences in model architecture, input representation or optimisation.

**Reading order:** bounded windows → grouped partitions → class support and CNN tensor → CNN learning and errors → matched Random Forest features and fitting → common Validation metrics → paired participant interpretation → research outputs.


**00. Colab Repository Setup**


In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")

if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


/content/human-movement-intelligence


**01. Imports and Research Environment**

Load existing Colab, plotting, model, source-data and evaluation utilities. Dataset-specific transformations and shared algorithms remain in their established modules.


In [2]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("wearable_activity_recognition")

# ----------------- 2. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 3. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

plot_style(font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11, legend_size=10, 
    figure_dpi=100, figure_size=(9, 4))

# --------------- 4. Machine Learning and Deep Learning Models ----------------
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, balanced_accuracy_score, ConfusionMatrixDisplay,
    confusion_matrix, classification_report,
)
import torch
from joblib import dump as dump_joblib

# ------------------ 5. HARTH and Shared Research Utilities -------------------
from core.output_download import download_outputs_from_colab
from harth_data import (
    HARTH_SUBJECTS, ACTIVITY_NAMES, SENSOR_COLUMNS,
    prepare_harth_window_cohort,
)
from core.data_splitting import split_group_indices
from core.feature_engineering import extract_signal_features
from core.temporal_cnn import TemporalCNN, train_temporal_cnn
from core.ml_baselines import (
    evaluate_predictions_by_subject, classification_class_report,
)
from core.group_cross_validation import compare_models_by_subject

# ------------------- 6. Training Device and Setup Summary --------------------
from core.computer_device import computer_device
device = computer_device()
DEVICE = device.type
output_dir = root / "wearable_activity_recognition" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
print("Research output directory:", output_dir)


Repository up to date.
Compute device and system information
OS: Linux 6.6.159+
CPU: x86_64
Installed RAM: 12.7 GiB
NVIDIA GPU: Tesla T4
Total VRAM: 15.00 GiB
PyTorch version: 2.14.1+cu130
CUDA device: Tesla T4
CUDA runtime: 13.0
cuDNN version: 92800
PyTorch total VRAM: 14.56 GiB
Selected compute device: cuda
Research output directory: /content/human-movement-intelligence/wearable_activity_recognition/output


**02. Create One Bounded Raw-Window Cohort**

Use the common time-safe HARTH source adapter once for the 29 Development participants and preserve raw sequences, original activity labels and source-row metadata. The maximum of 500 windows per participant is a computational restriction rather than class balancing. The resulting tensor is the reusable input for temporal modelling.


In [3]:
# ================= 02. Create One Bounded Raw-Window Cohort ==================

# ------------------- 1. Cohort and Sampling Configuration --------------------
RESERVED_TEST = {"S023", "S032"}
DEVELOPMENT_SUBJECTS = sorted(set(HARTH_SUBJECTS) - RESERVED_TEST)
TASK_CLASSES = tuple(range(1, 9))
WINDOW_CAP = 500

# ---------------------- 2. Prepare Matched Raw Windows -----------------------
X, y, metadata, overview = prepare_harth_window_cohort(
    subjects=DEVELOPMENT_SUBJECTS,
    data_dir=root / "data/raw/harth_v2",
    task_classes=TASK_CLASSES,
    max_windows_per_subject=WINDOW_CAP,
    window_size=100, step_size=50, sampling_rate=50,
)
if not (X.shape[0] == len(y) == len(metadata)):
    raise ValueError("Signal windows, labels and metadata must align.")
assert not (set(metadata["subject_id"]) & RESERVED_TEST)

# ------------------------ 3. Inspect Source Alignment ------------------------
if X.shape[1:] != (len(SENSOR_COLUMNS), 100):
    raise ValueError("Expected six acceleration channels and 100 samples.")
if not np.array_equal(metadata["label"].to_numpy(), y):
    raise ValueError("Window metadata and source labels are misaligned.")
print("Source-aligned temporal windows:", X.shape)

display(overview[[
    "subject_id", "eligible_windows", "sampled_windows",
    "observed_sampled_classes",
]])

# ----------------- 4. Export Source-Aligned Temporal Inputs ------------------
metadata.to_csv(output_dir / "04_bounded_window_metadata.csv", index=False)
overview.to_csv(output_dir / "04_bounded_cohort_summary.csv", index=False)
np.savez_compressed(
    output_dir / "04_bounded_signal_windows.npz",
    X_windows=X, y_windows=y, channel_names=np.asarray(SENSOR_COLUMNS),
    window_size=np.int64(100), step_size=np.int64(50),
)


Using cached HARTH data: /content/human-movement-intelligence/data/raw/harth_v2/S006.csv
HARTH subject S006: 408,709 rows, 6 sensors
Input signal: 408,709 samples, 6 channels
Candidate windows: 8,173
Retained windows: 7,441 | Shape: (7441, 6, 100)
Rejected candidates: 732
Using cached HARTH data: /content/human-movement-intelligence/data/raw/harth_v2/S008.csv
HARTH subject S008: 418,989 rows, 6 sensors
Input signal: 418,989 samples, 6 channels
Candidate windows: 8,378
Retained windows: 7,564 | Shape: (7564, 6, 100)
Rejected candidates: 814
Using cached HARTH data: /content/human-movement-intelligence/data/raw/harth_v2/S009.csv
HARTH subject S009: 154,464 rows, 6 sensors
Input signal: 154,464 samples, 6 channels
Candidate windows: 3,088
Retained windows: 2,885 | Shape: (2885, 6, 100)
Rejected candidates: 203
Using cached HARTH data: /content/human-movement-intelligence/data/raw/harth_v2/S010.csv
HARTH subject S010: 351,649 rows, 6 sensors
Input signal: 351,649 samples, 6 channels
Candid

/content/human-movement-intelligence/wearable_activity_recognition/code/src/harth_data.py:327: UserWarning: Unexpected HARTH activity codes: [10, 11, 17] (4,512 rows in S030.csv). The HARTH v2.0 activity table does not define these codes. Source codes retained without reinterpretation; resolve them before supervised training.
  data, channels = load_harth_subject(


HARTH subject S030: 245,994 rows, 6 sensors
Input signal: 245,994 samples, 6 channels
Candidate windows: 4,918
Retained windows: 3,335 | Shape: (3335, 6, 100)
Rejected candidates: 1,583
Downloaded 22,728,229 bytes to /content/human-movement-intelligence/data/raw/harth_v2/S031.csv


/content/human-movement-intelligence/wearable_activity_recognition/code/src/harth_data.py:327: UserWarning: Unexpected HARTH activity codes: [10, 11] (334 rows in S031.csv). The HARTH v2.0 activity table does not define these codes. Source codes retained without reinterpretation; resolve them before supervised training.
  data, channels = load_harth_subject(


HARTH subject S031: 269,560 rows, 6 sensors
Input signal: 269,560 samples, 6 channels
Candidate windows: 5,390
Retained windows: 4,411 | Shape: (4411, 6, 100)
Rejected candidates: 979
Downloaded 16,070,724 bytes to /content/human-movement-intelligence/data/raw/harth_v2/S033.csv
HARTH subject S033: 201,688 rows, 6 sensors
Input signal: 201,688 samples, 6 channels
Candidate windows: 4,032
Retained windows: 3,687 | Shape: (3687, 6, 100)
Rejected candidates: 345
Downloaded 15,031,875 bytes to /content/human-movement-intelligence/data/raw/harth_v2/S034.csv
HARTH subject S034: 188,202 rows, 6 sensors
Input signal: 188,202 samples, 6 channels
Candidate windows: 3,763
Retained windows: 3,351 | Shape: (3351, 6, 100)
Rejected candidates: 412
Downloaded 14,899,779 bytes to /content/human-movement-intelligence/data/raw/harth_v2/S035.csv
HARTH subject S035: 181,251 rows, 6 sensors
Input signal: 181,251 samples, 6 channels
Candidate windows: 3,624
Retained windows: 3,337 | Shape: (3337, 6, 100)
Reje

/content/human-movement-intelligence/wearable_activity_recognition/code/src/harth_data.py:327: UserWarning: Unexpected HARTH activity codes: [10] (1,422 rows in S037.csv). The HARTH v2.0 activity table does not define these codes. Source codes retained without reinterpretation; resolve them before supervised training.
  data, channels = load_harth_subject(


HARTH subject S037: 206,379 rows, 6 sensors
Input signal: 206,379 samples, 6 channels
Candidate windows: 4,126
Retained windows: 3,468 | Shape: (3468, 6, 100)
Rejected candidates: 658
Using cached HARTH data: /content/human-movement-intelligence/data/raw/harth_v2/S038.csv
HARTH subject S038: 184,992 rows, 6 sensors
Input signal: 184,992 samples, 6 channels
Candidate windows: 3,698
Retained windows: 3,355 | Shape: (3355, 6, 100)
Rejected candidates: 343
Temporal cohort: 29 subjects, X=(14201, 6, 100)
Source-aligned temporal windows: (14201, 6, 100)


,subject_id,eligible_windows,sampled_windows,observed_sampled_classes
0,S006,6708,500,7
1,S008,7392,500,5
2,S009,201,201,4
3,S010,6077,500,6
4,S012,7309,500,8
5,S013,6402,500,8
6,S014,6299,500,8
7,S015,7453,500,8
8,S016,6272,500,7
9,S017,6678,500,8


**03. Create Participant-Disjoint Data Partitions**

Fix a seed-42 split of whole participants into Train, Validation and unused Development. The splitting helper internally calls its third partition `test`; that label does not refer to reserved S023/S032 evaluation data. Both model families use the same row indices.


In [4]:
# ============== 03. Create Participant-Disjoint Data Partitions ==============

# ------------------- 1. Create Participant-Disjoint Split --------------------
parts = split_group_indices(
    metadata["subject_id"].to_numpy(),
    train_fraction=0.70, validation_fraction=0.20,
    test_fraction=0.10, random_state=42,
)
train_idx = parts["train"]
val_idx = parts["validation"]

# ------------------- 2. Verify Group and Signal Alignment --------------------
group_members = {
    name: set(metadata.iloc[rows]["subject_id"])
    for name, rows in parts.items()
}

assert not (group_members["train"] & group_members["validation"])
assert not (group_members["train"] & group_members["test"])
assert not (group_members["validation"] & group_members["test"])
X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

# --------------------- 3. Inspect Validation Provenance ----------------------
val_subject_ids = metadata.iloc[val_idx]["subject_id"].to_numpy()
if X_val.shape[0] != len(y_val) or len(y_val) != len(val_subject_ids):
    raise ValueError("Validation signal and group identities must align.")

display(pd.DataFrame([
    {
        "partition": "unused_development" if name == "test" else name,
        "subjects": len(group_members[name]),
        "windows": len(rows),
    },
     "IDs": ", ".join(sorted(group_members[name]))}
    for name, rows in parts.items()
]))


SyntaxError: closing parenthesis '}' does not match opening parenthesis '[' on line 29 (249607737.py, line 35)

**04. Inspect Activity Classes and CNN Tensor**

Display the eight-class Train/Validation support and verify the CNN's `(batch, channels, time)` input and eight-dimensional class-logit output. Missing Train classes invalidate this predefined target.


In [ ]:
# ================ 04. Inspect Activity Classes and CNN Tensor ================

# --------------------- 1. Check Training Class Coverage ----------------------
counts = pd.DataFrame({
    "train": pd.Series(y_train).value_counts(),
    "validation": pd.Series(y_val).value_counts(),
}).reindex(TASK_CLASSES, fill_value=0).fillna(0).astype(int)

counts.index = [f"{code}: {ACTIVITY_NAMES[code]}" for code in TASK_CLASSES]
display(counts)

# ------------------ 2. Inspect CNN Input and Output Shapes -------------------
assert (counts["train"] > 0).all(), "Training class coverage is incomplete."

model_preview = TemporalCNN(in_channels=6, n_classes=8)
example_output = model_preview(torch.from_numpy(X_train[:4]))

print("Example CNN output shape:", tuple(example_output.shape))
print("Untrained classifier output is logits, not probabilities.")


**05. Train the Five-Epoch Temporal CNN**

Train the original compact PyTorch 1D-CNN with batch size 128, learning rate `1e-3`, seed 42 and unweighted cross-entropy. Fit channel standardisation on Train signals only; retain the predeclared final epoch rather than selecting the best Validation epoch.


In [ ]:
# =================== 05. Train the Five-Epoch Temporal CNN ===================

# ------------------------ 1. Configure Model Training ------------------------
EPOCHS = 5
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
DEVICE = device.type

# ------------------- 2. Train CNN with Train-Only Scaling --------------------
cnn, history, cnn_prediction, scaler = train_temporal_cnn(
    X_train=X_train, y_train=y_train,
    X_validation=X_val, y_validation=y_val,
    classes=TASK_CLASSES,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE, seed=42, device=DEVICE,
    # Channel standardisation is estimated from Train data only.
)

# -------------------------- 3. Training Diagnostics --------------------------
display(history.round(4))
print("Channel means fitted from Train:", np.round(scaler["channel_mean"], 4))


**06. Visualise CNN Learning Curves**

Plot training loss and fixed eight-class Validation Macro F1 over the five epochs. Learning trends are within-Development diagnostics.


In [ ]:
# ===================== 06. Visualise CNN Learning Curves =====================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["epoch"], history["train_loss"], marker="o")
axes[0].set(title="Training loss", xlabel="Epoch", ylabel="Cross-entropy loss")
axes[0].grid(alpha=0.25)
axes[1].plot(history["epoch"], history["validation_macro_f1"], marker="o")
axes[1].set(title="Validation Macro F1", xlabel="Epoch",
            ylabel="Macro F1", ylim=(0, 1))
axes[1].grid(alpha=0.25)
fig.tight_layout()
save_figure(fig, output_dir / "04_cnn_training_validation_curves.png")
plt.show()


**07. Inspect CNN Activity-Level Errors**

Inspect the CNN's full classification report, including lower-support movement activities. The matched confusion matrices for both models are presented together in the model-comparison section.


In [ ]:
# =================== 07. Inspect CNN Activity-Level Errors ===================

# -------------------- 1. Validation Classification Report --------------------
report = classification_report(
    y_val, cnn_prediction,
    labels=TASK_CLASSES,
    target_names=[ACTIVITY_NAMES[c] for c in TASK_CLASSES],
    output_dict=True, zero_division=0,
)

report_table = pd.DataFrame(report).T
display(report_table.round(3))


**08. Extract Matched Random Forest Features**

Calculate five deterministic features for each of six channels from the exact same bounded windows. Train and Validation feature rows are selected using the existing raw-window indices.


In [ ]:
# ================ 08. Extract Matched Random Forest Features =================

# ------------------ 1. Extract Matched Statistical Features ------------------
feature_table = extract_signal_features(
    X, channel_names=SENSOR_COLUMNS,
    methods=["mean", "std", "rms", "peak_to_peak", "waveform_length"],
)
X_rf_train = feature_table.iloc[train_idx]
X_rf_val = feature_table.iloc[val_idx]

# ------------------------ 2. Verify Feature Alignment ------------------------
assert len(X_rf_train) == len(X_train)
assert len(X_rf_val) == len(X_val)
if feature_table.shape != (len(X), 30):
    raise ValueError("Matched Random Forest inputs must have 30 features.")

print("CNN Train:", X_train.shape, "| Validation:", X_val.shape)
print("RF Train:", X_rf_train.shape, "| Validation:", X_rf_val.shape)

# ------------------ 3. Export Aligned Statistical Features -------------------
pd.concat([
    metadata.add_prefix("window_").reset_index(drop=True),
    feature_table.reset_index(drop=True),
], axis=1).to_csv(
    output_dir / "04_bounded_statistical_features.csv", index=False
)


**09. Fit the Train-Only Random Forest**

Fit the defined 120-tree Random Forest with `min_samples_leaf=3`, balanced subsample class weighting and random seed 42. Validation labels do not influence fitting.


In [ ]:
# =================== 09. Fit the Train-Only Random Forest ====================

# -------------------- 1. Configure and Fit Random Forest ---------------------
rf = RandomForestClassifier(
    n_estimators=120, min_samples_leaf=3,
    class_weight="balanced_subsample", random_state=42, n_jobs=-1,
)
rf.fit(X_rf_train, y_train)

# -------------------- 2. Generate Validation Predictions ---------------------
rf_prediction = rf.predict(X_rf_val)
assert len(rf_prediction) == len(y_val)



**10. Compare Matched Validation Metrics**

Compute fixed eight-class Macro F1, balanced accuracy and activity-level precision, recall and F1 on the same Validation labels for both models. Interpret differences as a comparison of configured pipelines, not architecture alone.


In [ ]:
# ================== 10. Compare Matched Validation Metrics ===================

# ----------------------- 1. Compare Validation Metrics -----------------------
predictions = {
    "Random Forest": rf_prediction,
    "Temporal CNN": cnn_prediction,
}
comparison = pd.DataFrame([
    {"model": name,
     "macro_f1": f1_score(y_val, prediction, labels=TASK_CLASSES,
                         average="macro", zero_division=0),
     "balanced_accuracy": balanced_accuracy_score(y_val, prediction),
     "validation_windows": len(y_val)}
    for name, prediction in predictions.items()
])

display(comparison.round(4))

# -------------------------- 2. Inspect Per-Class F1 --------------------------
reports = []

for name, prediction in predictions.items():
    item = classification_class_report(
        y_val, prediction, classes=TASK_CLASSES,
        class_names=[ACTIVITY_NAMES[k] for k in TASK_CLASSES],
    )
    item["model"] = name
    reports.append(item)

class_table = pd.concat(reports, ignore_index=True)

display(class_table[[
    "model", "class", "support", "precision", "recall", "f1",
]].round(3))
fig, ax = plt.subplots(figsize=(12, 5))
positions = np.arange(8)

for i, name in enumerate(predictions):
    values = class_table.loc[class_table["model"] == name, "f1"].to_numpy()
    ax.bar(positions + (i - 0.5) * 0.36, values, width=0.34, label=name)

ax.set_xticks(positions, [ACTIVITY_NAMES[c] for c in TASK_CLASSES],
              rotation=35, ha="right")
ax.set(ylabel="Validation F1", title="Matched class-wise F1: RF versus CNN",
       ylim=(0, 1))
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
save_figure(fig, output_dir / "04_matched_per_class_f1.png")
plt.show()


**11. Compare Matched Confusion Matrices**

Use the same activity order for side-by-side Random Forest and temporal CNN confusion matrices. Both panels describe exactly the same held-out windows.


In [ ]:
# ================== 11. Compare Matched Confusion Matrices ===================
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, (name, prediction) in zip(axes, predictions.items()):
    matrix = confusion_matrix(y_val, prediction, labels=TASK_CLASSES)
    ConfusionMatrixDisplay(
        confusion_matrix=matrix,
        display_labels=[ACTIVITY_NAMES[c] for c in TASK_CLASSES],
    ).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
    ax.set_title(f"{name}: matched Validation")

fig.tight_layout()
save_figure(fig, output_dir / "04_matched_confusion_matrices.png")
plt.show()


**12. Inspect Paired Held-Out Participant Scores**

Evaluate participant-level fixed eight-class Macro F1 and display signed temporal CNN minus Random Forest differences for the same held-out people. The comparison is descriptive and does not treat overlapping windows as independent subjects.


In [ ]:
# ============== 12. Inspect Paired Held-Out Participant Scores ===============

# -------------------- 1. Compare Participant-Level Scores --------------------
subject_results = pd.concat([
    evaluate_predictions_by_subject(
        y_true=y_val, y_pred=prediction, subject_ids=val_subject_ids,
        classes=TASK_CLASSES, model_name=name,
    )
    for name, prediction in predictions.items()
], ignore_index=True)

display(subject_results.sort_values(["subject_id", "model"]).round(4))
paired, paired_summary = compare_models_by_subject(
    subject_results, model_a="Temporal CNN",
    model_b="Random Forest",
    score_column="macro_f1_fixed_task",
)

# ------------------- 2. Visualise Paired Model Differences -------------------
display(paired.round(4))
display(pd.DataFrame([paired_summary]).round(4))
fig, ax = plt.subplots(figsize=(10, 4))

difference = paired["difference_a_minus_b"]

ax.bar(paired["subject_id"], difference,
       color=["tab:blue" if d >= 0 else "tab:orange" for d in difference])
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Matched held-out subject Macro F1: CNN minus Random Forest",
       xlabel="Validation participant", ylabel="Paired Macro F1 difference")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
save_figure(fig, output_dir / "04_matched_participant_difference.png")
plt.show()


**13. Export Bounded and Matched Research Results**

Save full-precision cohort, split, CNN learning, class support, matched validation metrics, paired participant scores, source-aligned predictions and trained model artefacts under the stable `04_` prefix. The Colab packaging helper collects these research files for manual retrieval.


In [ ]:
# ============== 13. Export Bounded and Matched Research Results ==============

# ------------------------ 1. Export Matched Evidence -------------------------
split_summary = pd.DataFrame([
    {"partition": ("unused_development" if name == "test" else name),
     "subject_id": subject}
    for name, people in group_members.items()
    for subject in sorted(people)
])
split_summary.to_csv(
    output_dir / "04_development_partition_membership.csv", index=False
)
history.to_csv(output_dir / "04_cnn_training_history.csv", index=False)
comparison.to_csv(output_dir / "04_matched_model_validation_metrics.csv", index=False)
class_table.to_csv(output_dir / "04_matched_validation_class_metrics.csv", index=False)
subject_results.to_csv(
    output_dir / "04_matched_validation_subject_metrics.csv", index=False
)
paired.to_csv(output_dir / "04_matched_validation_paired_subjects.csv", index=False)
pd.DataFrame([paired_summary]).to_csv(
    output_dir / "04_matched_validation_paired_summary.csv", index=False
)
pd.DataFrame({
    "subject_id": val_subject_ids,
    "true_source_code": y_val,
    "rf_predicted_source_code": rf_prediction,
    "cnn_predicted_source_code": cnn_prediction,
}).to_csv(output_dir / "04_matched_validation_predictions.csv", index=False)
for name, prediction in predictions.items():
    safe_name = "random_forest" if name == "Random Forest" else "temporal_cnn"
    matrix = confusion_matrix(y_val, prediction, labels=TASK_CLASSES)
    pd.DataFrame(matrix, index=TASK_CLASSES, columns=TASK_CLASSES).rename_axis(
        "true_source_code"
    ).to_csv(output_dir / f"04_{safe_name}_validation_confusion.csv")

dump_joblib(rf, output_dir / "04_matched_random_forest.joblib")
torch.save({
    "model_state_dict": {k: v.detach().cpu() for k, v in cnn.state_dict().items()},
    "architecture": {"name": "TemporalCNN", "in_channels": 6,
                     "n_classes": 8, "hidden_channels": 32},
    "train_only_channel_mean": scaler["channel_mean"].tolist(),
    "train_only_channel_std": scaler["channel_std"].tolist(),
    "task_classes": list(TASK_CLASSES),
    "training": {
        "epochs": EPOCHS, "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE, "seed": 42,
        "class_weighting": "none",
    },
}, output_dir / "04_matched_temporal_cnn.pt")

# --------------------- 2. Export CNN Class Diagnostics -----------------------
counts.rename_axis("activity").reset_index().to_csv(
    output_dir / "04_train_validation_class_coverage.csv", index=False
)
report_table.rename_axis("class_or_metric").reset_index().to_csv(
    output_dir / "04_cnn_validation_classification_report.csv", index=False
)
pd.DataFrame({
    "channel": SENSOR_COLUMNS,
    "train_only_mean": scaler["channel_mean"],
    "train_only_std": scaler["channel_std"],
}).to_csv(output_dir / "04_cnn_train_only_scaling.csv", index=False)

# -------------------- 3. Package Current Research Outputs --------------------
download_outputs_from_colab(
    folder="wearable_activity_recognition/output", stage="04"
)


**14. Scientific Interpretation and Research Boundaries**

The bounded temporal experiment uses source windows as prediction units and participants as independent Train/Validation units. Selecting up to 500 eligible windows per person makes the protocol more tractable but changes activity support relative to the full Development cohort.

The CNN processes six original acceleration channels over 100 samples. Random Forest uses 30 deterministic statistics extracted from the same bounded source windows, with identical Train and Validation identities. The CNN uses Train-only scaling and a predeclared five-epoch endpoint. The Random Forest uses the original 120-tree, class-balanced configuration without feature scaling. The comparison therefore controls source identities but not input representation or optimisation.

Use aggregate metrics alongside class-specific confusion, activity support, and participant-equal paired differences. A six-person Validation comparison does not provide six independent repetitions of training; neighbouring, overlapping windows are correlated within each participant.

S023 and S032 remain outside all Development fitting and scoring. The separate full-Development five-fold classical analysis is not a directly comparable benchmark for this bounded single-partition experiment.

The `04_bounded_signal_windows.npz` and `04_bounded_window_metadata.csv` files describe reusable raw temporal inputs. The aligned `04_bounded_statistical_features.csv` supports classical comparisons but cannot replace raw windows for later CNN experiments.
